RANGE

In [1]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))


2.5.1+cu121
True
NVIDIA GeForce GTX 1650


In [1]:
import sys
sys.path.insert(0, "/D:\\Snowpole Detection\\ultralytics4channel-0a38736761a770f7f7dd80064e20b2d9624eda5b")  # parent of the ultralytics package

import ultralytics
from ultralytics import YOLO

print("Ultralytics module file:", ultralytics.__file__)


Ultralytics module file: d:\Snowpole Detection\cuda-env\Lib\site-packages\ultralytics\__init__.py


In [2]:
import cv2
import shutil
import yaml
import numpy as np
from pathlib import Path

ROOT = Path("SnowPole Detection A Comprehensive Dataset for Detection and Localization Using LiDAR Imaging in Nordic Winter Conditions\\SnowPole_Detection_Dataset")

RANGE_ROOT  = ROOT / "range"
LABELS_ROOT = ROOT / "labels"
OUT_ROOT    = ROOT / "hha_only"

(OUT_ROOT / "images").mkdir(parents=True, exist_ok=True)
(OUT_ROOT / "labels").mkdir(parents=True, exist_ok=True)

def compute_hha4(depth):
    depth = cv2.resize(depth, (1024, 1024))
    depth = depth.astype(np.float32)
    depth[depth == 0] = 1e-3

    # --- H: disparity ---
    disparity = 1.0 / depth
    disparity = cv2.normalize(disparity, None, 0, 255, cv2.NORM_MINMAX)

    # --- H: height (proxy) ---
    height = cv2.normalize(depth, None, 0, 255, cv2.NORM_MINMAX)

    # --- A: angle with gravity (approx via gradients) ---
    gx = cv2.Sobel(depth, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(depth, cv2.CV_32F, 0, 1, ksize=3)
    angle = np.arctan2(gy, gx)
    angle = cv2.normalize(angle, None, 0, 255, cv2.NORM_MINMAX)

    # --- 4th channel: raw depth (normalized) ---
    depth_norm = cv2.normalize(depth, None, 0, 255, cv2.NORM_MINMAX)

    hha4 = np.dstack([disparity, height, angle, depth_norm]).astype(np.uint8)
    return hha4

def make_split(split):
    img_out = OUT_ROOT / "images" / split
    lbl_out = OUT_ROOT / "labels" / split
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)

    for f in (LABELS_ROOT / split).glob("*.txt"):
        shutil.copy2(f, lbl_out / f.name)

    for p in (RANGE_ROOT / split).glob("*.*"):
        depth = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
        if depth is None:
            continue
        hha4 = compute_hha4(depth)
        cv2.imwrite(str(img_out / f"{p.stem}.png"), hha4)

for s in ["train", "valid", "test"]:
    make_split(s)

data_yaml = OUT_ROOT / "data.yaml"
cfg = {
    "path": str(OUT_ROOT),
    "train": "images/train",
    "val": "images/valid",
    "test": "images/test",
    "nc": 1,
    "names": ["snow_pole"],
    "channels": 4
}

with open(data_yaml, "w") as f:
    yaml.safe_dump(cfg, f)


In [3]:
from ultralytics import YOLO

model = YOLO("yolov8n.yaml")

model.train(
    data="SnowPole Detection A Comprehensive Dataset for Detection and Localization Using LiDAR Imaging in Nordic Winter Conditions\\SnowPole_Detection_Dataset\\hha_only\\data.yaml",
    imgsz=1024,
    epochs=300,
    patience=40,        # early stopping
    batch=2,
    device=0,
    project="Ablation_HHA",
    name="hha_only",
    amp=False,
    augment=False,
    workers=0
)

New https://pypi.org/project/ultralytics/8.3.246 available  Update with 'pip install -U ultralytics'
Ultralytics YOLOv8.2.5  Python-3.11.9 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce GTX 1650, 4096MiB)
engine\trainer: task=detect, mode=train, model=yolov8n.yaml, data=SnowPole Detection A Comprehensive Dataset for Detection and Localization Using LiDAR Imaging in Nordic Winter Conditions\SnowPole_Detection_Dataset\hha_only\data.yaml, epochs=300, time=None, patience=40, batch=2, imgsz=1024, save=True, save_period=-1, cache=False, device=0, workers=0, project=Ablation_HHA, name=hha_only3, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=False, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, s

d:\Snowpole Detection\cuda-env\Lib\site-packages\ultralytics\engine\trainer.py:262: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(enabled=self.amp)
train: Scanning D:\Snowpole Detection\SnowPole Detection A Comprehensive Dataset for Detection and Localization Using LiDAR Imaging in Nordic Winter Conditions\SnowPole_Detection_Dataset\hha_only\labels\train.cache... 1367 images, 0 backgrounds, 0 corrupt: 100%|██████████| 1367/1367 [00:00<?, ?it/s]
val: Scanning D:\Snowpole Detection\SnowPole Detection A Comprehensive Dataset for Detection and Localization Using LiDAR Imaging in Nordic Winter Conditions\SnowPole_Detection_Dataset\hha_only\labels\valid... 390 images, 0 backgrounds, 0 corrupt: 100%|██████████| 390/390 [00:05<00:00, 69.49it/s] 

val: New cache created: D:\Snowpole Detection\SnowPole Detection A Comprehensive Dataset for Detection and Localization Using LiDAR Imaging in Nordic Winter Conditions\SnowPole_Detection_Dataset\hha_only\labels\valid.cache


Plotting labels to Ablation_HHA\hha_only3\labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.002, momentum=0.9) with parameter groups 57 weight(decay=0.0), 64 weight(decay=0.0005), 63 bias(decay=0.0)
Image sizes 1024 train, 1024 val
Using 0 dataloader workers
Logging results to Ablation_HHA\hha_only3
Starting training for 300 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


      1/300      1.36G      2.936       20.2      1.468          4       1024: 100%|██████████| 684/684 [04:13<00:00,  2.69it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 98/98 [00:47<00:00,  2.05it/s]

...

                   all        390        789      0.719      0.607      0.681      0.246



300 epochs completed in 22.846 hours.
Optimizer stripped from Ablation_HHA\hha_only3\weights\last.pt, 6.4MB
Optimizer stripped from Ablation_HHA\hha_only3\weights\best.pt, 6.4MB

Validating Ablation_HHA\hha_only3\weights\best.pt...
Ultralytics YOLOv8.2.5  Python-3.11.9 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce GTX 1650, 4096MiB)
YOLOv8n summary (fused): 168 layers, 3005987 parameters, 0 gradients, 8.1 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 98/98 [00:35<00:00,  2.73it/s]


                   all        390        789      0.754      0.595       0.69       0.25
Speed: 0.9ms preprocess, 52.6ms inference, 0.0ms loss, 1.3ms postprocess per image
Results saved to Ablation_HHA\hha_only3


ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000001DBEA823BD0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0480

In [ ]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"


In [1]:
import torch
import cv2
import shutil
import yaml
import glob
import numpy as np
from pathlib import Path
import ultralytics
from ultralytics import YOLO

In [ ]:
# ===================== PATHS =====================
ROOT = Path(
    "D:\\Snowpole Detection\\SnowPole Detection A Comprehensive Dataset for Detection and Localization Using LiDAR Imaging in Nordic Winter Conditions"
    "\\SnowPole_Detection_Dataset"
)

COMB_ROOT   = ROOT / "combined_color"
RANGE_ROOT  = ROOT / "range"
OUT_ROOT   = ROOT / "1ch_range"
LABELS_ROOT = ROOT / "labels"

(OUT_ROOT / "images").mkdir(parents=True, exist_ok=True)
(OUT_ROOT / "labels").mkdir(parents=True, exist_ok=True)

In [ ]:
def preprocess_range(img):
    img = cv2.resize(img, (1024, 1024))
    img = cv2.equalizeHist(img)
    img = img.astype(np.float32) / 255.0
    return img


In [4]:
RGB_ROOT = ROOT / "3ch_rgb"

(RGB_ROOT / "images").mkdir(parents=True, exist_ok=True)
(RGB_ROOT / "labels").mkdir(parents=True, exist_ok=True)

def make_split(split):
    src_img_dir = COMB_ROOT / split
    dst_img_dir = RGB_ROOT / "images" / split
    dst_lbl_dir = RGB_ROOT / "labels" / split

    dst_img_dir.mkdir(parents=True, exist_ok=True)
    dst_lbl_dir.mkdir(parents=True, exist_ok=True)

    # copy labels
    for f in (LABELS_ROOT / split).glob("*.txt"):
        shutil.copy2(f, dst_lbl_dir / f.name)

    imgs = list(src_img_dir.glob("*.*"))
    print(f"{split}: {len(imgs)} images")

    for img_path in imgs:
        img = cv2.imread(str(img_path))
        if img is None:
            continue

        img = cv2.resize(img, (640, 640))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        outpath = dst_img_dir / f"{img_path.stem}.png"
        cv2.imwrite(str(outpath), cv2.cvtColor(img, cv2.COLOR_RGB2BGR))


In [5]:
for split in ["train", "valid", "test"]:
    make_split(split)


train: 1367 images
valid: 390 images
test: 197 images


In [6]:
for split in ["train", "valid", "test"]:
    paths = glob.glob(str(RGB_ROOT / f"images/{split}/*.png"))
    for p in paths:
        img = cv2.imread(p)
        if img is None or img.shape[2] != 3:
            print("BAD IMAGE:", p)


In [7]:
data_yaml = RGB_ROOT / "data.yaml"

cfg = {
    "path": str(RGB_ROOT),
    "train": "images/train",
    "val":   "images/valid",
    "test":  "images/test",
    "names": ["snow_pole"],
    "nc": 1
}

with open(data_yaml, "w") as f:
    yaml.safe_dump(cfg, f)


In [8]:
model = YOLO("yolov8n.yaml")

model.train(
    data=str(data_yaml),
    imgsz=640,
    epochs=50,
    batch=2,
    device=0,
    project="SnowPole_3ch",
    name="yolo_rgb",
    amp=False,
    workers=0
)


Ultralytics 8.3.250  Python-3.11.9 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce GTX 1650, 4096MiB)
engine\trainer: agnostic_nms=False, amp=False, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\Snowpole Detection\SnowPole Detection A Comprehensive Dataset for Detection and Localization Using LiDAR Imaging in Nordic Winter Conditions\SnowPole_Detection_Dataset\3ch_rgb\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.yaml, momentum=0.937, mos

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x0000021B537F5450>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0480